# Workers and context isolation

**Scenario:** a moderation queue fans out. One worker per flagged item, each writing a full policy
analysis, then one lead call that reads all of them and decides what to escalate.

It works on four items. On a real queue the lead costs more than every worker together, and
eventually stops fitting.

The rule is **a field report, not the field notes**. The worker reads everything. What crosses back
is the finding, not the reading.

## Mechanics

Fan out in LangGraph is `Send`. Tokens are the unit a model reads and bills in, so what a worker
returns is what the parent pays for.

| Piece | What it is | Why it matters here |
|---|---|---|
| `Send("node", payload)` | One worker run with its own input | The worker sees the payload, not the parent state |
| `add_conditional_edges(START, fn, ["node"])` | The fan out edge | `fn` returns a list of `Send` objects |
| `Annotated[list, operator.add]` | A reducer on a parent key | Without one, two workers writing one key raises |
| The worker return value | A dict of parent keys | This, and only this, reaches the parent |

Two workers writing the same unreduced key raises `InvalidUpdateError: at key 'notes': Can receive
only one value per step`. The parent state is shared. The worker state is not.

## The picture

![Four workers fanned out, returning either whole analyses or compact verdicts](images/workers-and-context.svg)

The solid path returns everything a worker wrote. The dotted path returns a verdict.

## The cost

The lead pays for every worker at once.

```
lead prompt tokens = its own instructions + sum(what each worker returned)
```

Worker count multiplies the second term, and worker verbosity multiplies it again.

## The failure

Four queue items and the analyst prompt most teams write first.

In [1]:
from vault import Usage, cost_of, get_client, load_env, model_for

load_env()
client = get_client("02-multi-agent-orchestration/02-workers-and-context-isolation")

QUEUE = [
    ("Q-1", "A live stream clip where a user shows prescription pills and jokes about selling them to classmates."),
    ("Q-2", "A comment thread calling a named journalist a traitor, with her home city and workplace posted."),
    ("Q-3", "A bystander video of a street fight, reposted by a news aggregator account with its own commentary."),
    ("Q-4", "A marketplace listing copied word for word from a brand site, by a seller with four prior takedowns."),
]

DEEP = ("You are a content moderation analyst. For the queue item write a full policy analysis: "
        "the policies engaged, the evidence for each, the aggravating and mitigating factors, the "
        "regional variation, and the recommended action with reasoning.")

LEAD = "You are the moderation queue lead. Summarise the queue and say which items to escalate."

The parent state. Workers run at the same time, so every key they write carries a reducer. The lead
writes alone, so its keys stay plain.

In [2]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Send


class Review(TypedDict):
    queue: list
    notes: Annotated[list, operator.add]
    worker_usage: Annotated[list, operator.add]
    lead_usage: object
    summary: str

The fan out returns one `Send` per item, carrying the only thing that worker sees.

In [3]:
def fan_out(state):
    """One worker per queue item, each with its own state."""
    return [Send("worker", {"item": item}) for item in state["queue"]]

Then the worker, written the obvious way. It hands back everything it wrote.

In [4]:
def review_deep(state):
    """Returns the whole analysis to the parent."""
    item_id, text = state["item"]
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=1500,
        messages=[{"role": "system", "content": DEEP},
                  {"role": "user", "content": f"{item_id}: {text}"}])
    return {"notes": [reply.choices[0].message.content or ""],
            "worker_usage": [Usage.from_response(reply)]}

The lead is one call over what the workers left behind.

In [5]:
def queue_lead(state):
    """One call that reads every note the workers returned."""
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=300,
        messages=[{"role": "system", "content": LEAD},
                  {"role": "user", "content": "\n\n".join(str(n) for n in state["notes"])}])
    return {"summary": reply.choices[0].message.content or "",
            "lead_usage": Usage.from_response(reply)}

One builder, so the two versions differ by their worker and nothing else.

In [6]:
def build(worker):
    """Same graph, different worker. START fans out, the lead closes."""
    graph = StateGraph(Review)
    graph.add_node("worker", worker)
    graph.add_node("lead", queue_lead)
    graph.add_conditional_edges(START, fan_out, ["worker"])
    graph.add_edge("worker", "lead")
    graph.add_edge("lead", END)
    return graph.compile()


EMPTY = {"queue": QUEUE, "notes": [], "worker_usage": [], "lead_usage": None, "summary": ""}

Run it with a budget on the lead call. Four items is a small queue, so instructions plus four
findings should fit.

In [7]:
BUDGET_TOKENS = 600

flood = build(review_deep).invoke(EMPTY)
deep_lead = flood["lead_usage"]

print(f"worker output tokens : {[u.completion_tokens for u in flood['worker_usage']]}")
print(f"lead prompt tokens   : {deep_lead.prompt_tokens}")
print(f"lead call cost       : {cost_of(deep_lead):.6f} USD")
assert deep_lead.prompt_tokens <= BUDGET_TOKENS, (
    f"the lead read {deep_lead.prompt_tokens} prompt tokens against a budget of {BUDGET_TOKENS}")

worker output tokens : [1034, 1054, 1500, 1463]
lead prompt tokens   : 5072
lead call cost       : 0.000627 USD


AssertionError: the lead read 5072 prompt tokens against a budget of 600

## The diagnosis

Look at the worker numbers. Each wrote a thousand tokens or more, because nothing told them not to.
One ran to its ceiling.

**The reducer did its job.** `notes` collected four analyses, as `operator.add` promises. The parent
grew because workers were allowed to write that much into it.

**The lead pays for all of it.** Its prompt is the sum of what came back, so worker count and worker
verbosity multiply. Four is the smallest interesting queue.

**Isolation was half done.** `Send` gave each worker its own state going in. Nothing bounded what
came out.

## The fix

The worker still reads everything and returns a verdict instead. First that verdict's shape,
enforced in code rather than requested in a prompt.

In [8]:
import json

MAX_FIELD = 90


def to_verdict(item_id, text):
    """All the parent is allowed to learn from one worker."""
    body = text.strip()
    if body.startswith("```"):
        body = body.split("```")[1].removeprefix("json").strip()
    parsed = json.loads(body)
    return {"item": item_id, "action": parsed["action"],
            "policy": str(parsed["policy"])[:MAX_FIELD],
            "why": str(parsed["why"])[:MAX_FIELD]}

The cap sits in the packing function, not the prompt, because a prompt is a request and a slice is a
promise. Then the worker, with a tighter brief.

In [9]:
TIGHT = ('You are a content moderation analyst. Reply with JSON only: '
         '{"action":"remove|restrict|allow|escalate","policy":str,"why":str} '
         'where "why" is at most 15 words.')


def review_tight(state):
    """Same reading, a report instead of the notes."""
    item_id, text = state["item"]
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=200,
        messages=[{"role": "system", "content": TIGHT},
                  {"role": "user", "content": f"{item_id}: {text}"}])
    return {"notes": [to_verdict(item_id, reply.choices[0].message.content or "")],
            "worker_usage": [Usage.from_response(reply)]}

Same graph, same lead call, same queue.

In [10]:
tight = build(review_tight).invoke(EMPTY)
tight_lead = tight["lead_usage"]

print(f"before: lead read {deep_lead.prompt_tokens:>5} prompt tokens, {cost_of(deep_lead):.6f} USD")
print(f"after : lead read {tight_lead.prompt_tokens:>5} prompt tokens, {cost_of(tight_lead):.6f} USD")
print(f"parent context is {deep_lead.prompt_tokens / tight_lead.prompt_tokens:.1f}x smaller")
print(f"verdict for Q-1: {tight['notes'][0]}")

before: lead read  5072 prompt tokens, 0.000627 USD
after : lead read   188 prompt tokens, 0.000092 USD
parent context is 27.0x smaller
verdict for Q-1: {'item': 'Q-1', 'action': 'remove', 'policy': 'Illegal acts and regulated goods', 'why': 'Promoting the sale of prescription pills is illegal and harmful.'}


## The gate

The regression is someone widening the verdict later, one field at a time. This plants a worker that
returns too much and asserts the parent never sees it.

In [11]:
def test_a_verbose_worker_cannot_widen_the_parent():
    shouting = json.dumps({"action": "remove", "policy": "x" * 4000, "why": "y" * 4000})
    packed = to_verdict("Q-9", shouting)
    size = len(json.dumps(packed))
    assert size < 400, f"one worker put {size} characters into the parent state"


test_a_verbose_worker_cannot_widen_the_parent()
print("gate holds: a worker reply of 8000 characters cannot widen the parent state")

gate holds: a worker reply of 8000 characters cannot widen the parent state


Delete the two slices in `to_verdict` and this test fails.

### Enterprise exploration

- The lead prompt grows with queue depth. At what depth does it stop fitting, and what happens to the
  next item?
- A capped verdict drops the reasoning. Where is the full analysis kept for an appeal, and what is the
  compliance cost of not keeping it?
- Workers run at the same time, one call each. What is the rate limit, and what does latency do when
  you hit it?
- Malformed JSON from one worker now raises inside a node. Fail the batch or drop the item?

### Key takeaways

- `Send` isolates a worker's input. Nothing isolates its output but you.
- A parent key written by two workers needs a reducer, or LangGraph refuses the update.
- The synthesis call pays for worker count multiplied by worker verbosity.
- Cap the report in code. A prompt asking for brevity is a request, not a bound.